# ╔══════════════════════════════════════════════════════════════════════╗
#    AttGraph - Subject-Independent / Dependent Evaluation
#    Based on: "EEG Emotion Recognition Using AttGraph"
# ╚══════════════════════════════════════════════════════════════════════╝

In [1]:
!pip uninstall \
plotnine mlxtend umap-learn \
pylibcugraph-cu12 imbalanced-learn \
libcugraph-cu12 dopamine-rl \
tsfresh cesium -y

!pip install torcheeg==1.1.2 scipy==1.12.0 numpy==1.26.4 torch_geometric -qqq

Found existing installation: plotnine 0.14.5
Uninstalling plotnine-0.14.5:
  Successfully uninstalled plotnine-0.14.5
Found existing installation: mlxtend 0.23.4
Uninstalling mlxtend-0.23.4:
  Successfully uninstalled mlxtend-0.23.4
Found existing installation: umap-learn 0.5.9.post2
Uninstalling umap-learn-0.5.9.post2:
  Successfully uninstalled umap-learn-0.5.9.post2
Found existing installation: pylibcugraph-cu12 25.6.0
Uninstalling pylibcugraph-cu12-25.6.0:
  Successfully uninstalled pylibcugraph-cu12-25.6.0
Found existing installation: imbalanced-learn 0.13.0
Uninstalling imbalanced-learn-0.13.0:
  Successfully uninstalled imbalanced-learn-0.13.0
Found existing installation: libcugraph-cu12 25.6.0
Uninstalling libcugraph-cu12-25.6.0:
  Successfully uninstalled libcugraph-cu12-25.6.0
Found existing installation: dopamine_rl 4.1.2
Uninstalling dopamine_rl-4.1.2:
  Successfully uninstalled dopamine_rl-4.1.2
Found existing installation: tsfresh 0.21.0
Uninstalling tsfresh-0.21.0:
  Suc

In [2]:
import os
import time
import numpy as np
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler, TensorDataset
from scipy import signal
from torcheeg import transforms
from torcheeg.datasets import SEEDIVFeatureDataset
import random
import warnings

In [3]:
warnings.filterwarnings('ignore')

In [4]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(100)

In [5]:
TRIAL_NUMBER = 18
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


In [6]:
# ── Preprocessing (Aligned with Paper) ────────────────────────────────
def BandPassFilter(eeg_data):
    # Paper uses 0.3 to 50 Hz
    b, a = signal.butter(4, Wn=[0.3, 50.0], btype='bandpass', fs=200)
    return signal.filtfilt(b, a, eeg_data, axis=-1)


def Notch(eeg_data):
    b, a = signal.iirnotch(w0=50.0, Q=30.0, fs=200)
    return signal.filtfilt(b, a, eeg_data, axis=-1)


In [7]:
LEFT_CH = list(range(0, 54, 2))
RIGHT_CH = list(range(1, 54, 2))


In [8]:
def pad_to_62(small_matrix):
    padded = np.zeros((62, 5))
    padded[:27, :] = small_matrix
    return padded


def calculate_de(coeffs):
    variance = np.var(coeffs, ddof=1)
    return 0.5 * np.log2(2 * np.pi * np.exp(1) * (variance + 1e-10))


def seed_iv_advanced_features_fn(x):
    num_channels = x.shape[0]
    num_bands = 5
    de_matrix = np.zeros((num_channels, num_bands))

    for ch in range(num_channels):
        coeffs = pywt.wavedec(x[ch], 'db4', level=5)
        for i in range(1, 6):
            de_matrix[ch, i - 1] = calculate_de(coeffs[i])

    dasm_small = de_matrix[LEFT_CH] - de_matrix[RIGHT_CH]
    dasm_62 = pad_to_62(dasm_small)

    return de_matrix.astype(np.float32)

    final_features = np.concatenate([de_matrix, dasm_62], axis=1)
    return final_features.astype(np.float32)


In [9]:
# Paper specifically uses 1s segments non-overlapping
SAMPLING_RATE = 400
SECONDS = 1
window = SECONDS * SAMPLING_RATE 
overlap_ratio = 0
step = int(window * (1 - overlap_ratio))
overlap_samples = window - step

In [10]:
t_transform = transforms.Compose(
    [
        transforms.Lambda(BandPassFilter),
        transforms.Lambda(Notch),
        transforms.BaselineRemoval(),
        transforms.Lambda(seed_iv_advanced_features_fn),
        # transforms.To2d(),
        transforms.ToTensor(),
    ]
)

In [11]:
io_path = f'./tmp_out/seed_iv_de_LDS_{TRIAL_NUMBER}'
# if os.path.exists(io_path):
#     shutil.rmtree(io_path)
root_path = './SEED-IV/eeg_feature_smooth/'
root_path = '/kaggle/input/seed-iv/eeg_feature_smooth'

In [12]:
dataset = SEEDIVFeatureDataset(
    io_path=io_path,
    root_path=root_path,
    feature=["de_LDS"],
    offline_transform=transforms.Compose(
        [
            transforms.BaselineRemoval(),
            transforms.To2d(),
            transforms.ToTensor(),
        ]
    ),
    label_transform=transforms.Compose(
        [
            transforms.Select("emotion"),
        ]
    ),
    online_transform=transforms.Compose([transforms.MeanStdNormalize()]),
    num_worker=4,
    verbose=False,
)

[2026-06-23 18:09:40] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_de_LDS_18.
[2026-06-23 18:09:40] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[2026-06-23 18:11:17] INFO (torcheeg/MainThread) ✅ | All processed EEG data has been cached to ./tmp_out/seed_iv_de_LDS_18.
[2026-06-23 18:11:17] INFO (torcheeg/MainThread) 😊 | Please set io_path to ./tmp_out/seed_iv_de_LDS_18 for the next run, to directly read from the cache if you wish to skip the data processing step.


In [13]:
x, y = dataset[0]

In [14]:
LABEL_REMAP = [0, 1, 2, 3]
# LABEL_REMAP = [0, 0, 1, 2]
# LABEL_REMAP = [0, 0, 0, 1]
# LABEL_REMAP = [0, 0, "UNREACHABLE", 1]
LABEL_REMAP = ["UNREACHABLE", 0, 0, 1]


def remap_labels(labels: np.ndarray) -> np.ndarray:
    return np.vectorize(lambda y: LABEL_REMAP[y])(labels)


In [15]:
IN_CHANNELS = x.shape[-1]
NUM_CLASSES = len(set(LABEL_REMAP))
NUM_NODES = 62
HIDDEN = 64
EPOCHS = 150  # Paper defaults to 40 epochs
BATCH_SIZE = 64
LR = 0.001  # Start lower to stabilize attention learning
WD = 1e-4
ALPHA = 0.1  # GRL strength — paper sets α = 0.1
PATIENCE = 20
EMOTION_LABELS = ['Neutral', 'Negative', 'Positive']


In [16]:
def load_subjects(subject_ids, meta_info, dataset):
    all_x, all_y = [], []
    for sid in subject_ids:
        subject_df = meta_info[meta_info['subject_id'] == sid]
        for idx in subject_df.index.tolist():
            x, y = dataset[idx]
            all_x.append(x.squeeze())
            all_y.append(y)
    all_x = torch.stack(all_x).numpy()
    all_y = remap_labels(np.array(all_y))
    return all_x, all_y


In [17]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# AttGraph Architecture Modules
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None


In [18]:
def grl(x, alpha):
    return GradientReversalLayer.apply(x, alpha)


In [19]:
class DynamicAttentionConv(nn.Module):
    """
    Multi-Dimensional Attention-Based Convolution Module
    A_att = softmax(H A H^T)
    Z = A_att * H
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.W = nn.Linear(in_channels, out_channels, bias=False)
        self.A = nn.Parameter(torch.Tensor(out_channels, out_channels))
        # Use smaller init to prevent large H·A·H^T magnitudes at start
        nn.init.xavier_uniform_(self.A, gain=0.1)
        self.scale = out_channels ** 0.5

    def forward(self, x_dense):
        H = self.W(x_dense)  # (B, E, F')

        # A_att = softmax(H · A · H^T / sqrt(F'))  — Equation (2) + scaling
        # The scaling by sqrt(F') is essential to prevent softmax saturation:
        # Without it, H·A·H^T has magnitude ~O(F'), making softmax near-one-hot
        # with vanishing gradients. Same principle as scaled dot-product attention
        # in Transformers (Vaswani et al., 2017).
        H_A = torch.matmul(H, self.A)
        H_A_Ht = torch.matmul(H_A, H.transpose(1, 2)) / self.scale

        A_att = F.softmax(H_A_Ht, dim=-1)
        A_att = (A_att + A_att.transpose(1, 2)) / 2.0  # Symmetrize — Equation (3)

        Z = torch.matmul(A_att, H)  # Equation (4)
        return Z


In [20]:
class GlobalAttentionPooling(nn.Module):
    """
    Attention-based readout: weighted sum over electrodes → (B, F')
    """

    def __init__(self, in_channels):
        super().__init__()
        self.attn = nn.Linear(in_channels, 1, bias=False)

    def forward(self, Z):
        omega = self.attn(Z)  # (B, E, 1)
        beta = F.softmax(omega, dim=1)  # Softmax across electrodes
        Z_prime = (beta * Z).sum(dim=1)  # (B, F')  — proper attention pooling
        return Z_prime


In [21]:
class AttGraph(nn.Module):
    def __init__(
        self,
        in_channels=IN_CHANNELS,
        hidden=HIDDEN,
        num_nodes=NUM_NODES,
        num_classes=NUM_CLASSES,
        num_subjects=15,  # For domain classifier
        dropout=0.3,
    ):
        super().__init__()
        self.conv1 = DynamicAttentionConv(in_channels, hidden)
        self.conv2 = DynamicAttentionConv(hidden, hidden)
        self.conv3 = DynamicAttentionConv(hidden, hidden)

        self.bn1 = nn.BatchNorm1d(hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.bn3 = nn.BatchNorm1d(hidden)

        self.dropout = nn.Dropout(dropout)
        self.pool = GlobalAttentionPooling(hidden)

        # 4-layer FC emotion classifier — Equation (6) in paper
        # ŷ = Softmax(W4 · σ(W3 · σ(W2 · σ(W1 · Flatten(Z')))))
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden),    # W1
            nn.ReLU(),                    # σ
            nn.Linear(hidden, hidden),    # W2
            nn.ReLU(),                    # σ
            nn.Linear(hidden, hidden),    # W3
            nn.ReLU(),                    # σ
            nn.Linear(hidden, num_classes),  # W4 (softmax applied in loss)
        )

        # Domain classifier with GRL — Section 4.2 in paper:
        # "FC with 100 hidden units, batch normalization, ReLU, LogSoftmax"
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden, 100),
            nn.BatchNorm1d(100),
            nn.ReLU(),
            nn.Linear(100, num_subjects),
            nn.LogSoftmax(dim=1),
        )

    def forward(self, x, alpha=0.0):
        # x shape: (B, 62, current_features)
        x = self.conv1(x)
        x = x.transpose(1, 2)
        x = self.bn1(x).transpose(1, 2)
        x = F.leaky_relu(x, 0.2)  # LeakyReLU prevents dead neurons after BN

        x = self.dropout(x)
        x = self.conv2(x)
        x = x.transpose(1, 2)
        x = self.bn2(x).transpose(1, 2)
        x = F.leaky_relu(x, 0.2)

        x = self.dropout(x)
        x = self.conv3(x)
        x = x.transpose(1, 2)
        x = self.bn3(x).transpose(1, 2)
        x = F.leaky_relu(x, 0.2)

        z_prime = self.pool(x)  # (B, hidden)

        # Emotion prediction
        emotion_out = self.classifier(z_prime)

        # Domain prediction via GRL — Equation (7)
        z_rev = grl(z_prime, alpha)
        domain_out = self.domain_classifier(z_rev)

        return emotion_out, domain_out


In [22]:
# --- MIXUP HELPER FUNCTIONS ---
def mixup_data(x, y, alpha=1.0):
    """
    Applies Mixup augmentation to a batch of data.
    All operations stay on the same device as the input tensors.

    Logic:
    1. Generate a mixing coefficient (lambda) from a Beta(alpha, alpha) distribution.
       - If alpha is close to 0, lambda is close to 0 or 1 (little mixing).
       - If alpha is large, lambda is close to 0.5 (strong mixing).
    2. Shuffle the batch indices to get a "partner" for every sample.
    3. Create a new "mixed" input: mixed_x = lambda * x + (1 - lambda) * x[shuffled_indices]

    Returns:
        mixed_x: The blended input tensor.
        y_a: The original labels.
        y_b: The labels of the shuffled samples.
        lam: The mixing coefficient used.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)  # create directly on GPU

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """
    Calculates the Mixup loss.

    Since the input was a mix of two samples (A and B) with weight lambda:
    Loss = lambda * Loss(pred, Label_A) + (1 - lambda) * Loss(pred, Label_B)
    """
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [23]:
train_transforms = transforms.Compose([
    transforms.RandomMask(p=0.1),
    transforms.RandomNoise(p=0.2),
])

In [24]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Training Loop (Emotion Classification Only)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def train_one_epoch(model, train_loader, optimizer, emotion_criterion, domain_criterion, alpha):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for x, y, d in train_loader:
        x, y, d = x.to(device), y.to(device), d.to(device)

        # Augmentation
        x = train_transforms(eeg=x)['eeg']

        # Apply Mixup (all on GPU)
        x, y_a, y_b, lam = mixup_data(x, y, alpha=0.2)

        optimizer.zero_grad()
        emotion_out, domain_out = model(x, alpha=alpha)

        # Combined loss: emotion classification + domain adversarial
        e_loss = mixup_criterion(emotion_criterion, emotion_out, y_a, y_b, lam)
        # e_loss = emotion_criterion(emotion_out, y)
        d_loss = domain_criterion(domain_out, d)
        loss = e_loss + d_loss

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct += (emotion_out.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / len(train_loader), correct / total


In [25]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    all_trues, all_preds = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        emotion_out, _ = model(x, alpha=0.0)
        pred_labels = emotion_out.argmax(1)

        correct += (pred_labels == y).sum().item()
        total += y.size(0)

        all_trues.extend(y.cpu().numpy())
        all_preds.extend(pred_labels.cpu().numpy())

    return correct / total, all_trues, all_preds


In [26]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LOSO Evaluation Protocol
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
meta_info = dataset.info
meta_info = dataset.info[dataset.info["emotion"] != 0]
all_subjects = sorted(dataset.info['subject_id'].unique())

loso_results = {}
loso_preds = {}


In [27]:
def make_tensor_dataset(features, labels, domain_labels=None):
    tensors = [
        torch.tensor(features, dtype=torch.float),
        torch.tensor(labels, dtype=torch.long),
    ]
    if domain_labels is not None:
        tensors.append(torch.tensor(domain_labels, dtype=torch.long))
    return TensorDataset(*tensors)


In [28]:
print('=' * 60)
print('   LOSO - AttGraph Architecture')
print('=' * 60)

# Build a mapping from subject_id to domain label index (0..N-1)
subject_to_domain = {sid: i for i, sid in enumerate(all_subjects)}
num_subjects = len(all_subjects)

for test_sid in all_subjects:
    train_sids = [s for s in all_subjects if s != test_sid]

    print(f'\n{"─" * 60}')
    print(f'  Test Subject  : {test_sid}')
    print(f'  Train Subjects: {train_sids}')
    print(f'{"─" * 60}')

    train_x, train_y = load_subjects(train_sids, meta_info, dataset)
    test_x, test_y = load_subjects([test_sid], meta_info, dataset)

    # Build domain labels for training subjects
    train_domain = []
    for sid in train_sids:
        subject_df = meta_info[meta_info['subject_id'] == sid]
        n_samples = len(subject_df)
        train_domain.extend([subject_to_domain[sid]] * n_samples)
    train_domain = np.array(train_domain)

    N_tr, C, B = train_x.shape
    N_te = test_x.shape[0]

    train_x_norm = train_x.astype(np.float32)
    test_x_norm = test_x.astype(np.float32)

    print(f'  Train: {N_tr} samples | Dist: {np.bincount(train_y, minlength=NUM_CLASSES)}')
    print(f'  Test : {N_te} samples | Dist: {np.bincount(test_y, minlength=NUM_CLASSES)}')

    train_ds = make_tensor_dataset(train_x_norm, train_y, train_domain)
    test_ds = make_tensor_dataset(test_x_norm, test_y)

    class_counts = np.bincount(train_y, minlength=NUM_CLASSES)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_w = [class_weights[y] for y in train_y]

    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = AttGraph(in_channels=B, num_subjects=num_subjects).to(device)

    emotion_criterion = nn.CrossEntropyLoss()
    domain_criterion = nn.NLLLoss()  # Pairs with LogSoftmax in domain classifier

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

    best_acc = 0.0
    best_trues, best_preds = [], []
    counter = 0

    for epoch in range(EPOCHS):
        startTime = time.time()

        loss, train_acc = train_one_epoch(
            model, train_loader, optimizer,
            emotion_criterion, domain_criterion, alpha=ALPHA
        )
        val_acc, trues, preds = evaluate(model, test_loader)

        elapsedTime = time.time() - startTime

        is_best_acc = False
        if val_acc > best_acc:
            best_acc = val_acc
            best_trues = trues.copy()
            best_preds = preds.copy()
            counter = 0
            is_best_acc = True
            torch.save(model.state_dict(), f"Best_AttGraph_loso_test{test_sid}.pt")
        else:
            counter += 1
            if counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch + 1}')
                counter = 0
                # break

        print(f'Epoch {epoch + 1:03d}: Train Acc={train_acc * 100:.2f}% | Test Acc={val_acc * 100:.2f}% | Loss: {loss:.4f} | Time: {elapsedTime:.2f}s {"[BEST]" if is_best_acc else ""}')

    print(f'\n  ✅ Subject {test_sid} Best Acc: {best_acc * 100:.2f}%')
    loso_results[test_sid] = best_acc * 100
    loso_preds[test_sid] = (best_trues, best_preds)


   LOSO - AttGraph Architecture

────────────────────────────────────────────────────────────
  Test Subject  : 1
  Train Subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
────────────────────────────────────────────────────────────
  Train: 25578 samples | Dist: [18172  7406     0]
  Test : 1827 samples | Dist: [1298  529    0]
Epoch 001: Train Acc=51.92% | Test Acc=45.70% | Loss: 3.2774 | Time: 4.42s [BEST]
Epoch 002: Train Acc=53.80% | Test Acc=28.95% | Loss: 3.2668 | Time: 3.42s 
Epoch 003: Train Acc=55.33% | Test Acc=34.81% | Loss: 3.2695 | Time: 3.42s 
Epoch 004: Train Acc=55.23% | Test Acc=29.78% | Loss: 3.2753 | Time: 3.50s 
Epoch 005: Train Acc=57.03% | Test Acc=42.26% | Loss: 3.2340 | Time: 3.38s 
Epoch 006: Train Acc=56.01% | Test Acc=71.10% | Loss: 3.2391 | Time: 3.43s [BEST]
Epoch 007: Train Acc=57.27% | Test Acc=63.55% | Loss: 3.2160 | Time: 3.54s 
Epoch 008: Train Acc=57.15% | Test Acc=42.47% | Loss: 3.2250 | Time: 3.50s 
Epoch 009: Train Acc=57.74% | Test Acc=3

In [29]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Final Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
all_accs = list(loso_results.values())
avg = np.mean(all_accs)
std = np.std(all_accs)

print(f'\n{"=" * 60}')
print('  LOSO FINAL - AttGraph Architecture')
print(f'{"=" * 60}')
print(f'  Average Accuracy : {avg:.2f}%')
print(f'  Std              : {std:.2f}%')
print(f'  Best  Subject    : {max(loso_results, key=loso_results.get)}  ({max(all_accs):.2f}%)')
print(f'  Worst Subject    : {min(loso_results, key=loso_results.get)}  ({min(all_accs):.2f}%)')
print(f'{"─" * 60}')
for sid, acc in loso_results.items():
    print(f'  Subject {sid:02d} : {acc:.2f}%')
print(f'{"=" * 60}')



  LOSO FINAL - AttGraph Architecture
  Average Accuracy : 77.32%
  Std              : 5.46%
  Best  Subject    : 14  (89.98%)
  Worst Subject    : 13  (70.50%)
────────────────────────────────────────────────────────────
  Subject 01 : 83.36%
  Subject 02 : 79.47%
  Subject 03 : 73.73%
  Subject 04 : 71.59%
  Subject 05 : 74.00%
  Subject 06 : 71.70%
  Subject 07 : 84.46%
  Subject 08 : 78.11%
  Subject 09 : 76.52%
  Subject 10 : 71.92%
  Subject 11 : 82.32%
  Subject 12 : 75.10%
  Subject 13 : 70.50%
  Subject 14 : 89.98%
  Subject 15 : 77.01%
